# 12.5 Cloud Deployment

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 12: Model Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-12-model-deployment/12.05_cloud-deployment.ipynb)

Companion code for Section 12.5 of the book. Run the cells in order from top to bottom.

**Example 12.5.1** · dockerfile  
Saved in this repo as `src/Dockerfile`.

```dockerfile
# Dockerfile - Multi-stage build for optimised ML service image
FROM python:3.11-slim as builder

WORKDIR /build

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc g++ && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ── Final image (slim, no build tools) ───────────────────────────
FROM python:3.11-slim

WORKDIR /app

# Copy installed packages from builder stage
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

# Copy application code and model
COPY main.py .
COPY model_pipeline.joblib .

# Non-root user for security
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

# Environment variables
ENV MODEL_PATH=/app/model_pipeline.joblib \
    PORT=8000 \
    WORKERS=4

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
    CMD python -c "import requests; requests.get('http://localhost:8000/health').raise_for_status()"

EXPOSE 8000

# Start with gunicorn (WSGI) or uvicorn (ASGI)
CMD ["uvicorn", "main:app",
     "--host", "0.0.0.0",
     "--port", "8000",
     "--workers", "4",
     "--log-level", "info"]
```

**Example 12.5.2** · text  
Saved in this repo as `src/requirements.txt`.

```text
# requirements.txt
fastapi==0.104.1
uvicorn[standard]==0.24.0
pydantic==2.5.0
scikit-learn==1.3.2
numpy==1.26.2
joblib==1.3.2
xgboost==2.0.2
lightgbm==4.1.0
python-multipart==0.0.6
httpx==0.25.2
prometheus-client==0.19.0
```

**Example 12.5.3** · yaml  
Saved in this repo as `src/docker-compose.yml`.

```yaml
# docker-compose.yml - Local development and testing
version: '3.8'

services:
  ml-api:
    build: .
    ports:
      - '8000:8000'
    environment:
      - MODEL_PATH=/app/model_pipeline.joblib
      - API_KEY=dev-secret-key
    volumes:
      - ./models:/app/models:ro   # Mount models directory as read-only
    restart: unless-stopped
    healthcheck:
      test: ['CMD', 'curl', '-f', 'http://localhost:8000/health']
      interval: 30s
      timeout: 10s
      retries: 3

  # Optional: Nginx reverse proxy with SSL termination
  nginx:
    image: nginx:alpine
    ports:
      - '80:80'
      - '443:443'
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
      - ./ssl:/etc/ssl:ro
    depends_on:
      - ml-api

  # Optional: Prometheus metrics collection
  prometheus:
    image: prom/prometheus:latest
    ports:
      - '9090:9090'
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
```

**Example 12.5.4** · bash  
Saved in this repo as `src/deploy_commands.sh`.

```bash
# deploy_commands.sh - Build, test, and deploy

# ── Build Docker image ────────────────────────────────────────────
docker build -t ml-cancer-classifier:v1.0 .
docker build -t ml-cancer-classifier:v1.0 --no-cache .  # Force rebuild

# ── Run locally ───────────────────────────────────────────────────
docker run -p 8000:8000 \
  -e API_KEY=my-secret-key \
  -e MODEL_PATH=/app/model_pipeline.joblib \
  ml-cancer-classifier:v1.0

# ── Test the container ────────────────────────────────────────────
curl http://localhost:8000/health

# ── Push to Docker Hub ────────────────────────────────────────────
docker tag ml-cancer-classifier:v1.0 yourusername/ml-api:v1.0
docker push yourusername/ml-api:v1.0

# ── Deploy to AWS ECS (Elastic Container Service) ─────────────────
# 1. Create ECR repository
aws ecr create-repository --repository-name ml-cancer-classifier

# 2. Authenticate Docker to ECR
aws ecr get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin \
  123456789.dkr.ecr.us-east-1.amazonaws.com

# 3. Push to ECR
docker tag ml-cancer-classifier:v1.0 \
  123456789.dkr.ecr.us-east-1.amazonaws.com/ml-cancer-classifier:v1.0
docker push 123456789.dkr.ecr.us-east-1.amazonaws.com/ml-cancer-classifier:v1.0

# 4. Deploy to ECS Fargate (serverless containers)
aws ecs update-service \
  --cluster ml-cluster \
  --service ml-api-service \
  --force-new-deployment

# ── Deploy to Google Cloud Run (simplest cloud option) ────────────
gcloud run deploy ml-api \
  --image gcr.io/PROJECT_ID/ml-cancer-classifier:v1.0 \
  --platform managed \
  --region us-central1 \
  --allow-unauthenticated \
  --memory 2Gi \
  --cpu 2 \
  --min-instances 1 \
  --max-instances 100 \
  --set-env-vars API_KEY=my-secret-key

# ── Deploy to Azure Container Instances ───────────────────────────
az container create \
  --resource-group ml-rg \
  --name ml-api \
  --image yourusername/ml-api:v1.0 \
  --cpu 2 --memory 4 \
  --ports 8000 \
  --environment-variables API_KEY=my-secret-key
```